# 06 - LoRA hyperparameter search

This notebook trains **8 LoRA configurations per pipeline** on the 2,768 training reports and scores every configuration on the 395 validation reports. It selects one configuration per pipeline on validation macro13. **The test set is not read in this notebook.**


## Search design

**Stage 1 - learning rate and epochs** (LoRA shape fixed at r=16, alpha=16):

| lr | epochs |
|---|---|
| 1e-4 | 1 |
| 1e-4 | 2 |
| 2e-4 | 1 |
| 2e-4 | 2 |

**Stage 2 - rank and alpha** at the stage-1 winning lr/epochs:

| r | alpha | alpha/r |
|---|---|---|
| 8 | 16 | 2 |
| 16 | 32 | 2 |
| 32 | 32 | 1 |
| 32 | 64 | 2 |

Stage 2 tests small/medium/large ranks at alpha=2r, plus alpha=r at rank 32 to test the scaling effect. 4 + 4 = 8 runs per pipeline, 16 in total.

The ranges are the standard LoRA bands: 1e-4 to 2e-4 is the usual learning-rate range for LoRA SFT, 1-2 epochs avoids overfitting 2,768 examples, and ranks 8-32 cover the usual capacity range. The old fixed configuration (r=16, alpha=16, lr=2e-4, 1 epoch) is one of the Stage-1 candidates, so the search also shows whether that choice was reasonable.





In [ ]:
!pip install -q unsloth==2026.9.9
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

import unsloth, torch, transformers, trl, peft, bitsandbytes
print("unsloth      ", unsloth.__version__)
print("transformers ", transformers.__version__)
print("torch        ", torch.__version__)
print("trl          ", trl.__version__)
print("GPU          ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Switch the Colab runtime to GPU (T4) before running this notebook."


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth       2026.9.9
transformers  5.5.0
torch         2.11.0+cu130
trl           0.24.0
GPU           Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, gc, inspect
from datetime import datetime, timezone
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/cxr_eval_data"
SEARCH = os.path.join(BASE, "lora_search")
os.makedirs(SEARCH, exist_ok=True)

need = ["sft_direct.jsonl", "sft_structure.jsonl", "prompt_templates.json", "train_set.csv", "val_set.csv"]
missing = [n for n in need if not os.path.exists(os.path.join(BASE, n))]
print(f"Found {len(need) - len(missing)}/{len(need)} required files in {BASE}")
if missing:
    for n in missing:
        print("  MISSING", n)
    raise FileNotFoundError("Run 05_sft_data.ipynb first (or upload the missing files), then rerun this cell.")

PC = ['No Finding','Enlarged Cardiomediastinum','Cardiomegaly','Lung Lesion','Lung Opacity','Edema',
      'Consolidation','Pneumonia','Atelectasis','Pneumothorax','Pleural Effusion','Pleural Other',
      'Fracture','Support Devices']
CLIN = [p for p in PC if p != 'No Finding']

PATH_COLS = {
    "lungs_and_airways.lung_opacity": "Lung Opacity", "lungs_and_airways.consolidation": "Consolidation",
    "lungs_and_airways.edema": "Edema", "lungs_and_airways.atelectasis": "Atelectasis",
    "lungs_and_airways.lung_lesion": "Lung Lesion", "lungs_and_airways.pneumonia": "Pneumonia",
    "pleura.pleural_effusion": "Pleural Effusion", "pleura.pneumothorax": "Pneumothorax",
    "pleura.pleural_other": "Pleural Other", "cardiovascular.cardiomegaly": "Cardiomegaly",
    "cardiovascular.enlarged_cardiomediastinum": "Enlarged Cardiomediastinum",
    "bones_and_chest_wall.fracture": "Fracture", "devices.support_devices": "Support Devices",
    "general.no_finding": "No Finding",
}
VMAP = {"present": "1", "absent": "0", "uncertain": "-1", "not_mentioned": "NaN", "nan": "NaN"}

def parse_direct(raw):
    parsed = {}
    for line in raw.strip().split("\n"):
        line = line.strip()
        if "=" in line:
            parts = line.split("=")
            if len(parts) == 2:
                name, value = parts[0].strip(), parts[1].strip()
                if name in PC and value in ("1", "0", "-1", "NaN", "nan"):
                    parsed[name] = value
    return [parsed.get(p, "NaN") for p in PC], len(parsed)

def parse_struct(raw):
    s, e = raw.find("{"), raw.rfind("}")
    if s == -1 or e <= s:
        return ["NaN"] * 14, 0
    try:
        data = json.loads(raw[s:e + 1])
    except Exception:
        return ["NaN"] * 14, 0
    out = {}
    for path, p in PATH_COLS.items():
        cur = data
        ok = True
        for k in path.split("."):
            if isinstance(cur, dict) and k in cur:
                cur = cur[k]
            else:
                ok = False
                break
        if ok and isinstance(cur, str) and cur.strip().lower() in VMAP:
            out[p] = VMAP[cur.strip().lower()]
    return [out.get(p, "NaN") for p in PC], len(out)

def parse_any(pipe, raw):
    return parse_direct(raw) if pipe == "direct" else parse_struct(raw)

with open(os.path.join(BASE, "prompt_templates.json"), encoding="utf-8") as f:
    TPL = json.load(f)
assert set(TPL) == {"direct", "structure"}

TRAIN = {}
for pipe in ("direct", "structure"):
    with open(os.path.join(BASE, f"sft_{pipe}.jsonl"), encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    assert len(rows) == 2768, (pipe, len(rows))
    TRAIN[pipe] = rows

train = pd.read_csv(os.path.join(BASE, "train_set.csv"))
val = pd.read_csv(os.path.join(BASE, "val_set.csv"))
assert len(train) == 2768 and len(val) == 395, (len(train), len(val))
assert set(train["id"]) & set(val["id"]) == set(), "train/val overlap"
for pipe in ("direct", "structure"):
    assert {r["id"] for r in TRAIN[pipe]} == set(train["id"]), pipe
print(f"train={len(train)}  val={len(val)}  SFT rows: direct={len(TRAIN['direct'])} structure={len(TRAIN['structure'])}")

def truth_arrays(df):
    out = {}
    for p in PC:
        t = df[p].map(lambda x: np.nan if pd.isna(x) else float(x)).to_numpy()
        out[p] = np.where(t == -1, np.nan, np.where(np.isnan(t), 0.0, t))
    return out

def macro_scores(values, truth):
    per = {}
    for j, p in enumerate(PC):
        t = truth[p]
        pr = np.array([1.0 if v[j] == "1" else 0.0 for v in values])
        tp = int(((t == 1) & (pr == 1)).sum())
        fp = int(((t == 0) & (pr == 1)).sum())
        fn = int(((t == 1) & (pr == 0)).sum())
        d = 2 * tp + fp + fn
        per[p] = 2 * tp / d if d else 0.0
    return {"macro13": float(np.mean([per[p] for p in CLIN])),
            "macro14": float(np.mean(list(per.values()))),
            "NoFinding": per["No Finding"]}

MANIFEST_COLS = ["pipeline", "config", "r", "alpha", "lr", "epochs", "seed", "steps", "train_loss",
                 "macro13", "macro14", "NoFinding", "valid14", "usable_any", "minutes",
                 "adapter_dir", "results_csv", "unsloth", "transformers", "torch", "done_utc"]
MANIFEST = os.path.join(SEARCH, "search_results.csv")

def load_manifest():
    if os.path.exists(MANIFEST):
        return pd.read_csv(MANIFEST)
    return pd.DataFrame(columns=MANIFEST_COLS)

def append_manifest(rec):
    m = load_manifest()
    if len(m):
        m = m[~((m["pipeline"] == rec["pipeline"]) & (m["config"] == rec["config"]))]
        m = pd.concat([m, pd.DataFrame([rec])], ignore_index=True)
        m.to_csv(MANIFEST, index=False)
    else:
        pd.DataFrame([rec]).to_csv(MANIFEST, index=False)

def config_name(pipe, cfg):
    return f"{pipe}_r{cfg['r']}_a{cfg['alpha']}_lr{cfg['lr']:g}_ep{cfg['epochs']}_s42"

def is_done(pipe, cfg):
    m = load_manifest()
    hit = m[(m["pipeline"] == pipe) & (m["config"] == config_name(pipe, cfg))]
    return len(hit) > 0 and os.path.exists(str(hit.iloc[-1]["results_csv"]))

print("Scorer ready: uncertain truth excluded, blank -> absent, only present positive, all reports scored.")


Mounted at /content/drive
Found 5/5 required files in /content/drive/MyDrive/cxr_eval_data
train=2768  val=395  SFT rows: direct=2768 structure=2768
Scorer ready: uncertain truth excluded, blank -> absent, only present positive, all reports scored.


## Training and inference functions

In [3]:
# Training and validation inference functions. Nothing runs until the smoke cell.
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

MAX_SEQ = 2048
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
INSTR = "<|start_header_id|>user<|end_header_id|>\n\n"
RESP = "<|start_header_id|>assistant<|end_header_id|>\n\n"
MAX_NEW = {"direct": 200, "structure": 300}
GEN_BATCH = 8

def load_lora(cfg):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="meta-llama/Llama-3.2-3B-Instruct",
        max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=cfg["r"], target_modules=TARGETS, lora_alpha=cfg["alpha"],
        lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    return model, tokenizer

def build_text(tokenizer, row):
    head = tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]}],
        tokenize=False, add_generation_prompt=True)
    return head + row["completion"] + tokenizer.eos_token

def train_one(cfg, pipe, tokenizer, model, outdir, smoke=False):
    if tokenizer.eos_token != "<|eot_id|>":
        raise RuntimeError(f"Unexpected EOS token: {tokenizer.eos_token!r}")
    texts = [build_text(tokenizer, r) for r in TRAIN[pipe]]
    ds = Dataset.from_list([{"text": t} for t in texts])
    args = SFTConfig(
        output_dir=outdir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=cfg["epochs"],
        max_steps=4 if smoke else -1,
        learning_rate=cfg["lr"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1 if smoke else 10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        save_strategy="no",
        report_to="none",
        dataset_text_field="text",
        max_length=MAX_SEQ,
        packing=False,
    )
    common = dict(model=model, train_dataset=ds, args=args)
    if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
        trainer = SFTTrainer(processing_class=tokenizer, **common)
    else:
        trainer = SFTTrainer(tokenizer=tokenizer, **common)
    _masked = train_on_responses_only(trainer, instruction_part=INSTR, response_part=RESP)
    if _masked is not None:
        trainer = _masked

    sample = trainer.train_dataset[0]
    kept = [t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100]
    decoded = tokenizer.decode(kept)
    if not decoded.lstrip().startswith(("No Finding=", "{")):
        raise RuntimeError("Response-only mask check failed. Unmasked tokens start with: " + repr(decoded[:120]))
    if smoke:
        print("  mask check passed; first unmasked tokens:", repr(decoded.lstrip()[:40]))

    stats = trainer.train()
    return trainer, stats

@torch.no_grad()
def predict(pipe, model, tokenizer, df):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    texts = []
    for report in df["full_report"]:
        p = TPL[pipe]["prefix"] + str(report) + TPL[pipe]["suffix"]
        texts.append(tokenizer.apply_chat_template(
            [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True))
    lens = [len(tokenizer(t)["input_ids"]) for t in texts]
    print(f"  prompt tokens max={max(lens)} p95={int(np.percentile(lens, 95))}")
    if max(lens) > MAX_SEQ - 100:
        raise RuntimeError(f"Prompt too long for max_seq_length={MAX_SEQ}: max {max(lens)}")
    raws = []
    for i in range(0, len(texts), GEN_BATCH):
        enc = tokenizer(texts[i:i + GEN_BATCH], return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_SEQ).to(model.device)
        out = model.generate(**enc, max_new_tokens=MAX_NEW[pipe],
                             do_sample=False, repetition_penalty=1.1)
        gen = out[:, enc["input_ids"].shape[1]:]
        raws.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
        print(f"  generated {len(raws)}/{len(texts)}", end="\r")
    print()
    return raws

def run_config(pipe, cfg, smoke=False):
    name = config_name(pipe, cfg)
    outdir = os.path.join(SEARCH, "_smoke" if smoke else pipe, name)
    os.makedirs(outdir, exist_ok=True)
    df = (val.head(8) if smoke else val).reset_index(drop=True)
    t0 = time.time()
    print(f"\n=== {name}{' (SMOKE)' if smoke else ''} ===")

    model, tokenizer = load_lora(cfg)
    trainer, stats = train_one(cfg, pipe, tokenizer, model, outdir, smoke=smoke)
    raws = predict(pipe, model, tokenizer, df)

    parsed = [parse_any(pipe, r) for r in raws]
    values = [x[0] for x in parsed]
    found = [x[1] for x in parsed]
    ms = macro_scores(values, truth_arrays(df))

    out_rows = []
    for i, raw in enumerate(raws):
        s = df.iloc[i]
        row = {"id": s["id"]}
        for p in PC:
            row[f"truth_{p}"] = s[p]
        for j, p in enumerate(PC):
            row[f"pred_{p}"] = values[i][j]
        row["raw"] = raw
        row["n_found"] = found[i]
        out_rows.append(row)
    res_csv = os.path.join(outdir, "val_predictions.csv")
    pd.DataFrame(out_rows).to_csv(res_csv, index=False)

    adapter_dir = os.path.join(outdir, "adapter")
    if not smoke:
        model.save_pretrained(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)

    minutes = (time.time() - t0) / 60
    import unsloth, transformers
    rec = {
        "pipeline": pipe, "config": name, "r": cfg["r"], "alpha": cfg["alpha"],
        "lr": cfg["lr"], "epochs": cfg["epochs"], "seed": 42,
        "steps": int(trainer.state.global_step),
        "train_loss": float(stats.training_loss),
        "macro13": ms["macro13"], "macro14": ms["macro14"], "NoFinding": ms["NoFinding"],
        "valid14": float(np.mean([f == 14 for f in found])),
        "usable_any": float(np.mean([f > 0 for f in found])),
        "minutes": minutes,
        "adapter_dir": None if smoke else adapter_dir,
        "results_csv": res_csv,
        "unsloth": unsloth.__version__,
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "done_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    if not smoke:
        append_manifest(rec)

    print(f"  macro13={ms['macro13']:.3f}  macro14={ms['macro14']:.3f}  "
          f"NoFind={ms['NoFinding']:.3f}  valid14={rec['valid14']*100:.1f}%  "
          f"usable={rec['usable_any']*100:.1f}%  ({minutes:.1f} min)")
    if smoke:
        print("  raw sample:", raws[0][:200].replace("\n", " | "))
        print("  smoke run: adapter not saved, manifest not updated")

    del trainer, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return rec


In [ ]:
# Smoke test 
for pipe in ["direct", "structure"]:
    run_config(pipe, dict(r=8, alpha=16, lr=1e-4, epochs=1), smoke=True)

print("\nSmoke test finished for both pipelines. If no error appeared, run the stage cells.")



=== direct_r8_a16_lr0.0001_ep1_s42 (SMOKE) ===
==((====))==  Unsloth 2026.9.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2768 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


... (16 outputs left out)

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.110468
2,0.139346
3,0.132887
4,0.108237


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  prompt tokens max=345 p95=324
  generated 8/8
  macro13=0.211  macro14=0.259  NoFind=0.889  valid14=87.5%  usable=100.0%  (1.7 min)
  raw sample: ```json | { |   "lungs_and_airways": { |     "lung_opacity": "present", |     "consolidation": "absent", |     "edema": "present", |     "atelectasis": "not_mentioned", |     "lung_lesion": "uncertain", |     "pneumo
  smoke run: adapter not saved, manifest not updated

Smoke test finished for both pipelines. If no error appeared, run the stage cells.


## Stage 1 - learning rate x epochs

In [ ]:
RUN_PIPELINES = ["direct", "structure"]

STAGE1 = [
    dict(r=16, alpha=16, lr=1e-4, epochs=1),
    dict(r=16, alpha=16, lr=1e-4, epochs=2),
    dict(r=16, alpha=16, lr=2e-4, epochs=1),
    dict(r=16, alpha=16, lr=2e-4, epochs=2),
]

for pipe in RUN_PIPELINES:
    for cfg in STAGE1:
        if is_done(pipe, cfg):
            print(f"skip {config_name(pipe, cfg)} - already finished")
            continue
        run_config(pipe, cfg)

m = load_manifest()
print("\nCurrent manifest:")
print(m[["pipeline", "config", "macro13", "macro14", "NoFinding", "valid14", "minutes"]].to_string(index=False))


skip direct_r16_a16_lr0.0001_ep1_s42 - already finished
skip direct_r16_a16_lr0.0001_ep2_s42 - already finished
skip direct_r16_a16_lr0.0002_ep1_s42 - already finished
skip direct_r16_a16_lr0.0002_ep2_s42 - already finished
skip structure_r16_a16_lr0.0001_ep1_s42 - already finished
skip structure_r16_a16_lr0.0001_ep2_s42 - already finished
skip structure_r16_a16_lr0.0002_ep1_s42 - already finished
skip structure_r16_a16_lr0.0002_ep2_s42 - already finished

Current manifest:
 pipeline                             config  macro13  macro14  NoFinding  valid14   minutes
   direct    direct_r16_a16_lr0.0001_ep1_s42 0.594669 0.552192   0.000000 1.000000 46.274149
   direct    direct_r16_a16_lr0.0001_ep2_s42 0.728244 0.731215   0.769841 1.000000 78.155960
   direct    direct_r16_a16_lr0.0002_ep1_s42 0.727868 0.733793   0.810811 1.000000 46.308329
   direct    direct_r16_a16_lr0.0002_ep2_s42 0.817884 0.825689   0.927152 1.000000 78.058705
structure structure_r16_a16_lr0.0001_ep1_s42 0.728059 0.

## Stage 1 result - choose the lr/epochs for stage 2

In [ ]:
m = load_manifest()
s1_names = [config_name(p, c) for p in RUN_PIPELINES for c in STAGE1]
m1 = m[m["config"].isin(s1_names)].copy()

if len(m1) == 0:
    print("No stage-1 runs yet.")
else:
    print(m1.sort_values(["pipeline", "macro13"], ascending=[True, False])[
        ["pipeline", "config", "macro13", "macro14", "NoFinding", "valid14", "minutes"]].to_string(index=False))

BEST1 = {}
for pipe in RUN_PIPELINES:
    sub = m1[m1["pipeline"] == pipe].copy()
    if len(sub) < len(STAGE1):
        print(f"\n{pipe}: stage 1 incomplete ({len(sub)}/{len(STAGE1)}). Stage 2 will skip this pipeline.")
        continue
    sub = sub.sort_values(["macro13", "r", "epochs"], ascending=[False, True, True])
    b = sub.iloc[0]
    BEST1[pipe] = (float(b["lr"]), int(b["epochs"]), float(b["macro13"]))
    print(f"\n{pipe}: stage-1 winner lr={b['lr']:g} epochs={int(b['epochs'])} val macro13={b['macro13']:.3f}")

print("\nBEST1 =", BEST1)


 pipeline                             config  macro13  macro14  NoFinding  valid14   minutes
   direct    direct_r16_a16_lr0.0002_ep2_s42 0.817884 0.825689   0.927152 1.000000 78.058705
   direct    direct_r16_a16_lr0.0001_ep2_s42 0.728244 0.731215   0.769841 1.000000 78.155960
   direct    direct_r16_a16_lr0.0002_ep1_s42 0.727868 0.733793   0.810811 1.000000 46.308329
   direct    direct_r16_a16_lr0.0001_ep1_s42 0.594669 0.552192   0.000000 1.000000 46.274149
structure structure_r16_a16_lr0.0002_ep2_s42 0.803662 0.811270   0.910180 1.000000 87.386724
structure structure_r16_a16_lr0.0001_ep2_s42 0.777467 0.780709   0.822857 0.972152 84.020698
structure structure_r16_a16_lr0.0002_ep1_s42 0.774884 0.777329   0.809117 0.918987 52.243849
structure structure_r16_a16_lr0.0001_ep1_s42 0.728059 0.730608   0.763736 0.741772 52.243358

direct: stage-1 winner lr=0.0002 epochs=2 val macro13=0.818

structure: stage-1 winner lr=0.0002 epochs=2 val macro13=0.804

BEST1 = {'direct': (0.0002, 2, 0.8178

## Stage 2 - rank x alpha at the winning lr/epochs

In [ ]:
if "BEST1" not in globals():
    raise RuntimeError("Run the stage-1 results cell first.")

STAGE2_PAIRS = [(8, 16), (16, 32), (32, 32), (32, 64)]

for pipe in RUN_PIPELINES:
    if pipe not in BEST1:
        print(f"skip {pipe}: stage 1 incomplete")
        continue
    lr, epochs, _ = BEST1[pipe]
    for r, alpha in STAGE2_PAIRS:
        cfg = dict(r=r, alpha=alpha, lr=lr, epochs=epochs)
        if is_done(pipe, cfg):
            print(f"skip {config_name(pipe, cfg)} - already finished")
            continue
        run_config(pipe, cfg)

m = load_manifest()
print("\nCurrent manifest:")
print(m[["pipeline", "config", "macro13", "macro14", "NoFinding", "valid14", "minutes"]].to_string(index=False))


skip direct_r8_a16_lr0.0002_ep2_s42 - already finished
skip direct_r16_a32_lr0.0002_ep2_s42 - already finished
skip direct_r32_a32_lr0.0002_ep2_s42 - already finished
skip direct_r32_a64_lr0.0002_ep2_s42 - already finished
skip structure_r8_a16_lr0.0002_ep2_s42 - already finished

=== structure_r16_a32_lr0.0002_ep2_s42 ===
==((====))==  Unsloth 2026.9.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2768 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


... (180 outputs left out)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  generated 395/395


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cxr_eval_data/lora_search/structure/structure_r32_a64_lr0.0002_ep2_s42/adapter/tokenizer_config.json.


  macro13=0.838  macro14=0.846  NoFind=0.959  valid14=100.0%  usable=100.0%  (80.8 min)

Current manifest:
 pipeline                             config  macro13  macro14  NoFinding  valid14   minutes
   direct    direct_r16_a16_lr0.0001_ep1_s42 0.594669 0.552192   0.000000 1.000000 46.274149
   direct    direct_r16_a16_lr0.0001_ep2_s42 0.728244 0.731215   0.769841 1.000000 78.155960
   direct    direct_r16_a16_lr0.0002_ep1_s42 0.727868 0.733793   0.810811 1.000000 46.308329
   direct    direct_r16_a16_lr0.0002_ep2_s42 0.817884 0.825689   0.927152 1.000000 78.058705
structure structure_r16_a16_lr0.0001_ep1_s42 0.728059 0.730608   0.763736 0.741772 52.243358
structure structure_r16_a16_lr0.0001_ep2_s42 0.777467 0.780709   0.822857 0.972152 84.020698
structure structure_r16_a16_lr0.0002_ep1_s42 0.774884 0.777329   0.809117 0.918987 52.243849
structure structure_r16_a16_lr0.0002_ep2_s42 0.803662 0.811270   0.910180 1.000000 87.386724
   direct     direct_r8_a16_lr0.0002_ep2_s42 0.807272 0.

## Selection (validation macro13)

In [ ]:
# Final selection per pipeline (validation macro13) 
m = load_manifest()
selected = {}

for pipe in RUN_PIPELINES:
    sub = m[m["pipeline"] == pipe].copy()
    if len(sub) < 8:
        print(f"\n{pipe}: {len(sub)}/8 configurations finished - selection skipped until the search is complete.")
        continue
    sub = sub.sort_values(["macro13", "r", "epochs", "alpha"], ascending=[False, True, True, True])
    cols = ["config", "r", "alpha", "lr", "epochs", "macro13", "macro14", "NoFinding", "valid14"]
    print(f"\n{pipe} - top 3 of 8 configurations")
    print(sub[cols].head(3).to_string(index=False))
    b = sub.iloc[0]
    selected[pipe] = {
        "config": str(b["config"]), "r": int(b["r"]), "alpha": int(b["alpha"]),
        "lr": float(b["lr"]), "epochs": int(b["epochs"]), "seed": 42,
        "val_macro13": float(b["macro13"]), "val_macro14": float(b["macro14"]),
        "val_NoFinding": float(b["NoFinding"]), "val_valid14": float(b["valid14"]),
        "adapter_dir": str(b["adapter_dir"]),
        "selected_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "selection_rule": "highest val macro13; ties: lower rank, fewer epochs, lower alpha",
    }

out_path = os.path.join(SEARCH, "selected_configs.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(selected, f, indent=1)
print("\nWrote", out_path)
print("Remember: the selected validation score is optimistic. The unbiased fine-tuned result is the single test run in 07.")



direct - top 3 of 8 configurations
                         config  r  alpha     lr  epochs  macro13  macro14  NoFinding  valid14
direct_r16_a32_lr0.0002_ep2_s42 16     32 0.0002       2 0.844913 0.853431   0.964169      1.0
direct_r32_a32_lr0.0002_ep2_s42 32     32 0.0002       2 0.844610 0.852196   0.950820      1.0
direct_r32_a64_lr0.0002_ep2_s42 32     64 0.0002       2 0.837878 0.845257   0.941176      1.0

structure - top 3 of 8 configurations
                            config  r  alpha     lr  epochs  macro13  macro14  NoFinding  valid14
structure_r32_a32_lr0.0002_ep2_s42 32     32 0.0002       2 0.875457 0.880804   0.950311 0.997468
structure_r16_a32_lr0.0002_ep2_s42 16     32 0.0002       2 0.843822 0.851407   0.950000 0.974684
structure_r32_a64_lr0.0002_ep2_s42 32     64 0.0002       2 0.837693 0.846375   0.959248 1.000000

Wrote /content/drive/MyDrive/cxr_eval_data/lora_search/selected_configs.json
Remember: the selected validation score is optimistic. The unbiased fine-tu